In [1]:
!pip install mcp pywin32 FastMCP

  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
Using cached packaging-26.0-py3-none-any.whl (74 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 23.2
    Uninstalling packaging-23.2:
      Successfully uninstalled packaging-23.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 0.1.53 requires packaging<24.0,>=23.2, but you have packaging 26.0 which is incompatible.
langchain-mcp 0.2.1 requires langchain-core~=0.3.37, but you have langchain-core 0.1.53 which is incompatible.
langchain-openai 0.0.6 requires openai<2.0.0,>=1.10.0, but you have openai 2.31.0 which is incompatible.
langgraph-checkpoint 4.0.1 requires langchain-core>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.1.53 which is incompatible.


In [2]:
# Cell 2: Create the server code (weather.py)
# We'll write the server file to disk so the subprocess can run it.

server_code = """
from fastmcp import FastMCP
import requests

app = FastMCP("WeatherServer")

@app.tool()
def get_weather(city: str) -> str:
    \"""Get current weather for a city.\"""
    response = requests.get(f"https://wttr.in/{city}?format=3")
    return response.text

if __name__ == "__main__":
   app.run(transport="sse", host="127.0.0.1", port=8000)

"""

with open("weather.py", "w") as f:
    f.write(server_code)

print("weather.py written")

weather.py written


!python weather.py

In [3]:
!uvicorn WeatherServer:app --host 127.0.0.1 --port 8001

ERROR:    Error loading ASGI app. Could not import module "WeatherServer".


In [4]:
from mcp.client.sse import sse_client
from mcp.client.session import ClientSession

async def main():
    async with sse_client("http://localhost:8101/sse") as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Tools:", tools)
            # Call tool
            result = await session.call_tool("get_weather", {"city": "Pune"})
            print("Result:", result)

await main()

  + Exception Group Traceback (most recent call last):
  |   File "D:\Ethans\Python\AI-Automation\myenv\Lib\site-packages\IPython\core\interactiveshell.py", line 3745, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\panka\AppData\Local\Temp\ipykernel_4944\1005749590.py", line 14, in <module>
  |     await main()
  |   File "C:\Users\panka\AppData\Local\Temp\ipykernel_4944\1005749590.py", line 5, in main
  |     async with sse_client("http://localhost:8000/sse") as (read, write):
  |                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\contextlib.py", line 210, in __aenter__
  |     return await anext(self.gen)
  |            ^^^^^^^^^^^^^^^^^^^^^
  |   File "D:\Ethans\Python\AI-Automation\myenv\Lib\site-packages\mcp\client\sse.py", line 63, in sse_client
  |     async with anyio.create_task_group() as tg:
  |                